# Multi-Stage Queries in SLayer

Most semantic layers parameterize a single GROUP BY. But what if you need to aggregate *an aggregation* — like averaging monthly revenue across stores, or bucketing users by a computed metric?

SLayer handles this naturally: every query implies a dataset (via auto-introspection of its result columns), so you can chain queries together. This notebook demonstrates two multi-stage patterns with working code.

See also: [Query Lists](../../concepts/queries.md#query-lists) | [Query extensions](../../concepts/queries.md#query-extensions) | [Creating Datasets from Queries](../../concepts/models.md#creating-datasets-from-queries)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## Queries as Datasets

In most semantic layers, queries and datasets are entirely separate concepts. In SLayer, every query resolves to SQL, and SLayer's [auto-introspection](../../concepts/ingestion.md) can generate a dataset from any SQL SELECT query. Put these together: **every query automatically implies a dataset**.

When a query becomes a virtual dataset, its result columns become dimensions, and numeric columns also get auto-generated aggregation metrics. This means you can use one query's output as the input to another — no special syntax needed.

## Example 1: Average Monthly Revenue per Store

**Goal:** What is the average monthly revenue for each store?

This is a two-stage query:
1. **Inner query:** Total revenue grouped by store and month
2. **Outer query:** Average the monthly revenue, grouped by store

The inner query's `order_total_sum` result column becomes a dimension on the virtual dataset, which we can then aggregate with `avg(order_total_sum)` in the outer query.

In [2]:
# Stage 1: Monthly revenue by store
inner_result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["sum(orders.order_total)"],
        "dimensions": ["stores.name"],
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "month"}],
        "order": [{"field": "orders.ordered_at", "direction": "asc"}],
        "limit": 10,
    }
)

print("Inner query result (first 10 rows):")
print(f"{'Month':<12} {'Store':<20} {'Revenue':>12}")
print("-" * 46)
for row in inner_result.data:
    month = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} {store:<20} ${rev:>11,.2f}")

Inner query result (first 10 rows):
Month        Store                     Revenue
----------------------------------------------
2023-11      Philadelphia         $   6,476.29
2023-12      Philadelphia         $  22,484.60
2024-01      Philadelphia         $  26,483.61
2024-02      Philadelphia         $  33,301.17
2024-03      Philadelphia         $  43,013.17
2024-04      Philadelphia         $  47,661.36
2024-05      Philadelphia         $  50,103.48
2024-05      Brooklyn             $   1,337.44
2024-06      Philadelphia         $  51,692.54
2024-06      Brooklyn             $  38,624.47


Now we chain the two queries together. We give the inner query a `name` and pass both queries as a list to `engine.execute_sync()`. The outer query references the inner query's name as its `population`:

In [3]:
# Two-stage query: average monthly revenue per store
result = engine.execute_sync(
    query=[
        {
            "name": "monthly_store_revenue",
            "population": "orders",
            "metrics": ["sum(orders.order_total)"],
            "dimensions": ["stores.name"],
            "time_dimensions": [{"field": "orders.ordered_at", "granularity": "month"}],
        },
        {
            "population": "monthly_store_revenue",
            "metrics": ["avg(monthly_store_revenue.order_total_sum)"],
            "dimensions": ["monthly_store_revenue.stores__name"],
            "order": [{"field": "monthly_store_revenue.order_total_sum_avg", "direction": "desc"}],
        },
    ]
)

print("Average monthly revenue by store:")
print(f"{'Store':<20} {'Avg Monthly Revenue':>20}")
print("-" * 42)
for row in result.data:
    store = row["monthly_store_revenue.stores__name"]
    avg_rev = row["monthly_store_revenue.order_total_sum_avg"]
    print(f"{store:<20} ${avg_rev:>19,.2f}")

Average monthly revenue by store:
Store                 Avg Monthly Revenue
------------------------------------------
Brooklyn             $          90,488.96
Chicago              $          69,752.90
San Francisco        $          61,562.99
Philadelphia         $          57,710.03
New Orleans          $          24,963.51


**What happened under the hood:**

1. SLayer executed the inner query (`monthly_store_revenue`), producing rows of (store, month, revenue)
2. It auto-introspected those results to create a virtual dataset with:
   - Dimensions: `stores__name`, `ordered_at`, `order_total_sum`
   - Metrics: one per numeric column (e.g., `order_total_sum`), queryable with aggregations like `avg(order_total_sum)`, `min(order_total_sum)`, etc. `count(*)` is always available.
3. The outer query used this virtual dataset, grouping by `stores__name` and requesting `order_total_sum_avg`

The virtual dataset is a self-contained table — it no longer has the relationships the source dataset had. That's why the dimension is called `stores__name` (using `__` to encode the original relationship path) rather than `stores.name` (which would imply a relationship to a `stores` dataset that doesn't exist on the virtual dataset).

No special multi-stage syntax was needed — just a named query and a reference to it.

In [4]:
print("Generated SQL:")  # NOSONAR(S1192) — tutorial cells are intentionally self-contained; pulling this label into a top-level constant would force readers to scroll between cells
print(result.sql)

Generated SQL:
WITH __slayer_stage_monthly_store_revenue AS (
  SELECT
    _stage_inner."orders.stores.name" AS "stores__name",
    _stage_inner."orders.ordered_at" AS "ordered_at",
    _stage_inner."orders.order_total_sum" AS "order_total_sum"
  FROM (
    SELECT
      stores.name AS "orders.stores.name",
      DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
      SUM(orders.order_total) AS "orders.order_total_sum"
    FROM orders AS orders
    LEFT JOIN stores AS stores
      ON orders.store_id = stores.id
    GROUP BY
      stores.name,
      DATE_TRUNC('MONTH', orders.ordered_at)
  ) AS _stage_inner
)
SELECT
  monthly_store_revenue.stores__name AS "monthly_store_revenue.stores__name",
  CAST(AVG(monthly_store_revenue.order_total_sum) AS DOUBLE) AS "monthly_store_revenue.order_total_sum_avg"
FROM __slayer_stage_monthly_store_revenue AS monthly_store_revenue
GROUP BY
  monthly_store_revenue.stores__name
ORDER BY
  "monthly_store_revenue.order_total_sum_avg" DESC


## Example 2: Orders by Customer Activity Bucket

**Goal:** How many orders and how much revenue comes from low, medium, and high-activity customers?

This is a two-stage pattern with a relationship:
1. **Inner query:** Compute total order count per customer
2. **Outer query:** Join the inner result to `orders`, add a CASE-based bucket dimension via a [query extension](../../concepts/queries.md#query-extensions), and group by the bucket

The bucketing uses an [inline dimension](../../concepts/queries.md#query-extensions) — a CASE expression that categorizes each customer's total order count into Low / Medium / High.

In [5]:
# Stage 1: Total orders per customer
inner_result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(orders.*)"],
        "dimensions": ["orders.customer_id"],
        "order": [{"field": "orders._count", "direction": "desc"}],
        "limit": 10,
    }
)

print("Top 10 customers by order count:")
print(f"{'Customer ID':<40} {'Orders':>10}")
print("-" * 52)
for row in inner_result.data:
    cust = row["orders.customer_id"]
    count = row["orders._count"]
    print(f"{cust:<40} {count:>10,}")

Top 10 customers by order count:
Customer ID                                  Orders
----------------------------------------------------
38eceda2-fa2e-47b0-94ce-ae267b449537            793
9e83842d-05bf-45c0-9e64-927fda81f87e            791
8bfd532d-939d-4027-a17c-0b5bea6f1f20            789
e028b102-2257-49e2-bf59-be7842369f72            789
fea53fe3-63d7-4cc5-b5f7-bfc7c18ccbe7            785
4b38abde-764b-4cba-824b-6261d2b52893            784
f8b1139e-c8a7-4ede-aad5-9abafd968691            783
0fdf2f8d-bcce-4d39-8e7d-ff24a6626b9c            782
73d16509-6bc5-4b09-a8e1-8b6669604b84            780
f9198354-cce2-462a-961e-1afbfb489ae7            780


Now we chain the queries. The outer query's `extensions` entry for `orders` **joins** the inner query (customer order counts) to the `orders` dataset via `customer_id`; the query's own `dimensions` list **defines the bucket** as a computed dimension: a CASE expression over the customer's total order count.

Since the bucket expression reads a column of the joined sub-query, it references it as `customer_activity._count` (the sub-query name followed by the column).

In [6]:
# Two-stage query: orders grouped by customer activity bucket
result = engine.execute_sync(
    query=[
        {
            "name": "customer_activity",
            "population": "orders",
            "metrics": ["count(orders.*)"],
            "dimensions": ["orders.customer_id"],
        },
        {
            "population": "orders",
            "extensions": [{
                "dataset": "orders",
                "relationships": [
                    {
                        "to": "customer_activity",
                        "from_columns": ["customer_id"], "to_columns": ["customer_id"],
                    }
                ],
            }],
            "metrics": ["count(orders.*)", "sum(orders.order_total)"],
            "dimensions": [{
                "field": "CASE WHEN customer_activity._count >= 500 THEN 'High' WHEN customer_activity._count >= 200 THEN 'Medium' ELSE 'Low' END",
                "alias": "activity_bucket",
            }],
            "order": [{"field": "orders.order_total_sum", "direction": "desc"}],
        },
    ]
)

print("Orders by customer activity bucket:")
print(f"{'Bucket':<12} {'Orders':>10} {'Revenue':>14}")
print("-" * 38)
for row in result.data:
    bucket = row["orders.activity_bucket"]
    count = row["orders._count"]
    rev = row["orders.order_total_sum"]
    print(f"{bucket:<12} {count:>10,} ${rev:>13,.2f}")

Orders by customer activity bucket:
Bucket           Orders        Revenue
--------------------------------------
Medium          329,615 $ 3,532,170.58
Low             128,513 $ 2,539,637.83
High            204,444 $ 1,275,735.12


In [7]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
WITH __slayer_stage_customer_activity AS (
  SELECT
    _stage_inner."orders.customer_id" AS "customer_id",
    _stage_inner."orders._count" AS "_count"
  FROM (
    SELECT
      orders.customer_id AS "orders.customer_id",
      COUNT(*) AS "orders._count"
    FROM orders AS orders
    GROUP BY
      orders.customer_id
  ) AS _stage_inner
)
SELECT
  CASE
    WHEN customer_activity._count >= 500
    THEN 'High'
    WHEN customer_activity._count >= 200
    THEN 'Medium'
    ELSE 'Low'
  END AS "orders.activity_bucket",
  COUNT(*) AS "orders._count",
  SUM(orders.order_total) AS "orders.order_total_sum"
FROM orders AS orders
LEFT JOIN __slayer_stage_customer_activity AS customer_activity
  ON orders.customer_id = customer_activity.customer_id
GROUP BY
  CASE
    WHEN customer_activity._count >= 500
    THEN 'High'
    WHEN customer_activity._count >= 200
    THEN 'Medium'
    ELSE 'Low'
  END
ORDER BY
  "orders.order_total_sum" DESC


## Example 3: Parallel Siblings Joined in a Final Stage

Stages in a query list are not restricted to a linear pipeline. Any stage may use any *prior* named sibling stage as its `population` or as a `relationships.to`, so several rollups can run in parallel and feed a single final stage.

Below: stage 1 aggregates orders per customer; stage 2 — itself a non-final named stage — joins that result back to `customers` to attach the per-customer total to each row; stage 3 takes the max.

In [8]:
# Three-stage DAG: kpis (per-customer rollup) → tagged (joined back to customers) → final (max)
result = engine.execute_sync(
    query=[
        {
            "name": "kpis",
            "population": "orders",
            "dimensions": ["orders.customer_id"],
            "metrics": ["sum(orders.order_total)"],
        },
        {
            "name": "tagged",
            "population": "customers",
            "extensions": [{
                "dataset": "customers",
                "relationships": [
                    {
                        "to": "kpis",
                        "from_columns": ["id"], "to_columns": ["customer_id"],
                    }
                ],
            }],
            "dimensions": ["customers.name", "kpis.order_total_sum"],
        },
        {
            "population": "tagged",
            "metrics": ["max(tagged.kpis__order_total_sum)"],
        },
    ]
)

print("Max per-customer order total:")
for row in result.data:
    for k, v in row.items():
        print(f"  {k} = {v:,.2f}")

Max per-customer order total:
  tagged.kpis__order_total_sum_max = 13,554.84


**What's new vs. Example 2:** in Example 2 only the *final* stage joined a named sibling. Here, the **non-final** stage `tagged` joins the prior named stage `kpis`, so the structure is a DAG with three stages instead of a two-step chain.

Forward references and self references are rejected with a clear error — a stage may only resolve to stages defined earlier in the list.

In [9]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
WITH __slayer_stage_kpis AS (
  SELECT
    _stage_inner."orders.customer_id" AS "customer_id",
    _stage_inner."orders.order_total_sum" AS "order_total_sum"
  FROM (
    SELECT
      orders.customer_id AS "orders.customer_id",
      SUM(orders.order_total) AS "orders.order_total_sum"
    FROM orders AS orders
    GROUP BY
      orders.customer_id
  ) AS _stage_inner
), __slayer_stage_tagged AS (
  SELECT
    _stage_inner."customers.name" AS "name",
    _stage_inner."customers.kpis.order_total_sum" AS "kpis__order_total_sum"
  FROM (
    SELECT
      customers.name AS "customers.name",
      kpis.order_total_sum AS "customers.kpis.order_total_sum"
    FROM customers AS customers
    LEFT JOIN __slayer_stage_kpis AS kpis
      ON customers.id = kpis.customer_id
    GROUP BY
      customers.name,
      kpis.order_total_sum
  ) AS _stage_inner
)
SELECT
  CAST(MAX(tagged.kpis__order_total_sum) AS DOUBLE) AS "tagged.kpis__order_total_sum_max"
FROM __slayer_stage_tagged AS tag

## Summary

Multi-stage queries in SLayer require no special syntax — they emerge naturally from two features:

| Feature | Role in Multi-Stage Queries |
|---------|----------------------------|
| **Query-as-dataset** | Every query's result auto-generates a virtual dataset with dimensions and metrics |
| **Query lists** | Pass `[inner, outer]` to `engine.execute_sync()` — inner queries are named, outer references them |
| **Auto-generated metrics** | Numeric columns from inner queries get one metric each, queryable with any aggregation |
| **Extension relationships** | Join a sub-query's virtual dataset to any stored dataset at query time |
| **Inline dimensions** | Add CASE expressions or other computed dimensions via `extensions` |

See the [Query Lists docs](../../concepts/queries.md#query-lists) and [query extension docs](../../concepts/queries.md#query-extensions) for the full reference.